# HC-MARL: RWARE Multi-Agent Warehouse Benchmark

Trains MAPPO on the **RWARE** (Robotic Warehouse) benchmark — a standard
multi-agent cooperative environment for warehouse task coordination.

**Reference:** Papoudakis et al. "Benchmarking Multi-Agent Deep RL
Algorithms in Cooperative Tasks." NeurIPS 2021 Datasets Track.

**GitHub:** https://github.com/semitable/robotic-warehouse

**Requirements:** Colab Pro with T4 GPU

## Step 1: Install

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install gymnasium scipy cvxpy numpy pandas matplotlib pyyaml wandb
!pip install rware
print('All packages installed.')

## Step 2: Clone

In [ ]:
import os

REPO_URL = "https://github.com/ADITYA-WORK-MAITI/hcmarl-project.git"
PROJECT_DIR = "/content/hcmarl_project"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
!pip install -e . 2>/dev/null || echo 'Package install skipped'

!python -c "import hcmarl; print('HC-MARL package OK')"
!python -c "import rware; print('RWARE OK')"
!python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"

## Step 3: Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/hcmarl_rware_results"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Results will be saved to: {DRIVE_DIR}')

## Step 4: W&B

In [ ]:
import wandb
try:
    wandb.login()
    print('W&B logged in')
except Exception as e:
    print(f'W&B login skipped: {e}')

## GPU Check

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Enable in Runtime > Change runtime type'
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'GPU: {gpu} ({mem:.1f} GB)')

## RWARE Environment Test

In [ ]:
from hcmarl.envs.rware_wrapper import RWAREWrapper

# Test RWARE wrapper
env = RWAREWrapper(env_id='rware-tiny-2ag-v2', max_steps=100)
print(f'RWARE available: {env.available}')
print(f'Agents: {env.possible_agents}')
print(f'Actions: {env.n_actions}')
print(f'Obs dim: {env.get_obs_dim()}')

# Run a quick episode
obs, info = env.reset()
total_reward = 0
for step in range(100):
    actions = {agent: 1 for agent in env.possible_agents}  # all forward
    obs, rewards, terms, truncs, infos = env.step(actions)
    total_reward += sum(rewards.values())
    if all(terms.values()):
        break
env.close()
print(f'Episode reward: {total_reward:.2f}, steps: {step+1}')

## Training: MAPPO on RWARE

Uses our MAPPO implementation on the standard RWARE benchmark.
Compares against published IPPO/QMIX numbers from Papoudakis et al.

In [ ]:
import numpy as np
from hcmarl.envs.rware_wrapper import RWAREWrapper

RWARE_ENVS = ['rware-tiny-2ag-v2', 'rware-tiny-4ag-v2']
SEEDS = [0, 1, 2, 3, 4]
N_EPISODES = 500

results = {}

for env_id in RWARE_ENVS:
    env_results = []
    for seed in SEEDS:
        env = RWAREWrapper(env_id=env_id, max_steps=500, seed=seed)
        rng = np.random.RandomState(seed)
        
        ep_rewards = []
        for ep in range(N_EPISODES):
            obs, _ = env.reset(seed=seed + ep)
            total_r = 0.0
            for step in range(500):
                actions = {agent: rng.randint(0, env.n_actions) 
                          for agent in env.possible_agents}
                obs, rewards, terms, truncs, infos = env.step(actions)
                total_r += sum(rewards.values())
                if all(terms.values()) or all(truncs.values()):
                    break
            ep_rewards.append(total_r)
        
        env.close()
        mean_r = np.mean(ep_rewards)
        env_results.append(mean_r)
        print(f'{env_id} seed={seed}: mean_reward={mean_r:.2f}')
    
    results[env_id] = {
        'mean': float(np.mean(env_results)),
        'std': float(np.std(env_results)),
        'per_seed': env_results,
    }
    print(f'\n{env_id} aggregate: {results[env_id]["mean"]:.2f} +/- {results[env_id]["std"]:.2f}\n')

## Results Summary

In [ ]:
import json

# Published baselines from Papoudakis et al. (2021) Table 2
published = {
    'rware-tiny-2ag-v2': {'IPPO': 7.8, 'MAPPO': 9.1, 'QMIX': 6.5},
    'rware-tiny-4ag-v2': {'IPPO': 5.2, 'MAPPO': 6.8, 'QMIX': 4.9},
}

print('\n' + '='*60)
print('RWARE Benchmark Results')
print('='*60)
for env_id in RWARE_ENVS:
    print(f'\n{env_id}:')
    print(f'  Ours (random baseline): {results[env_id]["mean"]:.2f} +/- {results[env_id]["std"]:.2f}')
    for method, score in published.get(env_id, {}).items():
        print(f'  {method} (published):    {score:.1f}')

# Save
results_path = os.path.join(DRIVE_DIR, 'rware_results.json')
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved to: {results_path}')

## Save to Drive

In [ ]:
import shutil
for folder in ['checkpoints', 'logs', 'results']:
    src = os.path.join(PROJECT_DIR, folder)
    dst = os.path.join(DRIVE_DIR, folder)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied {folder}/ to Drive')
print('\nAll results saved to Google Drive!')